# Model the Air Quality Index with a gaussian process.

In [96]:
import os
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from MuyGPyS._test.sampler import print_results
from MuyGPyS.neighbors import NN_Wrapper
from MuyGPyS.gp import MuyGPS
from MuyGPyS.gp.deformation import Isotropy, l2, Anisotropy
from MuyGPyS.gp.hyperparameter import AnalyticScale, Parameter, VectorParameter
from MuyGPyS.gp.kernels import Matern
from MuyGPyS.gp.noise import HomoscedasticNoise
from MuyGPyS.optimize import Bayes_optimize, L_BFGS_B_optimize
from MuyGPyS.optimize.batch import sample_batch
from MuyGPyS.optimize.loss import lool_fn, mse_fn, pseudo_huber_fn, looph_fn
import pandas as pd
import time

# Import and setup data

In [97]:
BASE_DIR = os.path.abspath("..") + "\\"
aqi_data = np.genfromtxt(BASE_DIR + "data\\biggest_day_aqi_coords_scaled_new.csv", delimiter=',', skip_header=1)
#print(aqi_data[:10])

In [98]:
#randomly split the data into features and responces, and those in turn into training and testing paritions.
aqi_x, aqi_y = aqi_data[:, :-1], aqi_data[:, -1]
aqi_x_train, aqi_x_test, aqi_y_train, aqi_y_test = train_test_split(aqi_x, aqi_y, test_size=0.1, random_state=20)

#use the mean of the training points to provide basic de-meaning of the points.
aqi_y_mean = aqi_y_train.mean()
aqi_y_train = aqi_y_train - aqi_y_mean

# Nearest Neighbor and Batches

In [99]:
#setup nearest nieghbors
nn_count = 30
nbrs_lookup = NN_Wrapper(aqi_x_train, nn_count, nn_method="exact", algorithm="ball_tree")


In [100]:
#for the sake of making use of batches, will use a batch value of 1500. this will include all of the data points.
batch_count = 1500
#get the indices of the training points and the indices of each points neighbors.
batch_indices, batch_nn_indices = sample_batch(
    nbrs_lookup, batch_count, len(aqi_x_train)
)

# setting and optimizing hyperparamters

In [101]:
#create our unoptimized muyGP object. set the ranges we want the hyperparameters to be optimized between.
aqi_muygps = MuyGPS(
    kernel=Matern(
        smoothness=Parameter("log_sample", (0.5, 2.5)),
        deformation=Isotropy(
            l2,
            length_scale= Parameter("log_sample", (0.01, 0.20)),
        ),
    ),
    noise=HomoscedasticNoise("sample", (0.003, 0.25)),
    scale=AnalyticScale(),
)

In [102]:
#use the indices of the training points, and of the training points neighbors, together with the points' coordinates and values
#to calculate the crosswise distances between points and the pairwise distances between the neighbors
#and also the actual value of each point, and each point's neighbor.
(
    batch_crosswise_dists,
    batch_pairwise_dists,
    batch_ys,
    batch_nn_ys,
) = aqi_muygps.make_train_tensors(
    batch_indices,
    batch_nn_indices,
    aqi_x_train,
    aqi_y_train,
)

In [103]:
optimize_start = time.time()
#optimize the parameters for 15 random initilization runs, and then 25 optimizing runs.
aqi_muygps_optimized = Bayes_optimize(
    aqi_muygps,
    batch_ys,
    batch_nn_ys,
    batch_crosswise_dists,
    batch_pairwise_dists,
    loss_fn=lool_fn,
    verbose=True,
    random_state=24,
    init_points=20,
    n_iter=30,
)

# aqi_muygps_optimized = L_BFGS_B_optimize(
#     aqi_muygps,
#     batch_ys,
#     batch_nn_ys,
#     batch_crosswise_dists,
#     batch_pairwise_dists,
#     loss_fn=mse_fn,
#     verbose=True,
# )
optimize_end = time.time()
optimize_length = optimize_end - optimize_start
print(f"Gaussian Process optimization took {optimize_length} seconds")

parameters to be optimized: ['smoothness', 'noise']
bounds: [[0.5   2.5  ]
 [0.003 0.25 ]]
initial x0: [0.53976111 0.02867428]
|   iter    |  target   | smooth... |   noise   |
-------------------------------------------------
| 1         | -7982.478 | 0.5397611 | 0.0286742 |
| 2         | -8512.659 | 2.4200346 | 0.1757794 |
| 3         | -16791.40 | 2.4997345 | 0.0573566 |
| 4         | -8128.280 | 1.2221127 | 0.1857407 |
| 5         | -13141.57 | 2.4929114 | 0.0811377 |
| 6         | -8685.800 | 0.7730891 | 0.0978430 |
| 7         | -11048.14 | 1.1410385 | 0.0935044 |
| 8         | -7689.033 | 1.9193031 | 0.2253351 |
| 9         | -15054.65 | 1.5682308 | 0.0640815 |
| 10        | -9394.518 | 1.8436131 | 0.1417470 |
| 11        | -7686.754 | 1.5851197 | 0.2236815 |
| 12        | -13386.98 | 2.1855590 | 0.0785851 |
| 13        | -8582.093 | 1.7623395 | 0.1710189 |
| 14        | -7720.952 | 2.4408551 | 0.2237110 |
| 15        | -8844.898 | 2.3848517 | 0.1616296 |
| 16        | -16099.48

In [104]:
#also optimize the scale
aqi_muygps_optimized = aqi_muygps_optimized.optimize_scale(
    batch_pairwise_dists,
    batch_nn_ys
)

# Inference

In [105]:
test_count = aqi_x_test.shape[0]
#get the indices of all the test points
test_indices = np.arange(test_count)
#get the indices of the neighbors for each of those test points as well
test_nn_indices, _ = nbrs_lookup.get_nns(aqi_x_test)

In [106]:
#create tesors to store the crosswise distances and pairwise distances between points and the point's neighbors.
#as well as the actual values for each points neighbors.
(
    test_crosswise_dists,
    test_pairwise_dists,
    test_nn_ys,
) = aqi_muygps.make_predict_tensors(
    test_indices,
    test_nn_indices,
    aqi_x_test,
    aqi_x_train,
    aqi_y_train,
)

In [107]:
#calculate kernels to store the crosswise covariance and pairwise covariance. these are used to know how much a points neighbors influence it.
kcross = aqi_muygps_optimized.kernel(test_crosswise_dists)
kin = aqi_muygps_optimized.kernel(test_pairwise_dists)

In [108]:
#Take the crosswise covariance and pairwise covariance to know how the neighbors effect each point,
#and then use the values of the points to calculate the actual value based on the neighbors influence.
predictions = aqi_muygps_optimized.posterior_mean(kin, kcross, test_nn_ys)
#because mean was removed from the datapoints before, need to add it back in.
predictions = predictions + aqi_y_mean
#calculate the variances, how much the guess is spread, meaning uncertainty.
variances = aqi_muygps_optimized.posterior_variance(kin, kcross)
#get the range that values need to be in in order to fall within 95% certainty.
confidence_intervals = np.sqrt(variances) * 1.96
#coverage is the proportion of guesses that differ from the true response by no more than the confidence interval size.
coverage = np.count_nonzero(np.abs(aqi_y_test - predictions) < confidence_intervals) / test_count

In [109]:
#print the results of the model
print_results(
    aqi_y_test, ("optimized", aqi_muygps_optimized, predictions, variances, confidence_intervals, coverage)
)

name,smoothness,length scale,noise variance,variance scale,rmse,mean variance,mean confidence interval,coverage
optimized,0.500000,1.000000,0.250000,634.191262,13.328939,17.530200,8.113801,0.559633


In [110]:
print(predictions[:10])
print(variances[:10])

[36.80754872 45.86689945 48.42969651 51.66355192 48.2541804  51.06592897
 63.31818354 31.36527944 50.36112252 43.99955362]
[29.9790002  13.3635623  12.05628439 17.11085286 23.65569271 11.64675525
 20.08054627 19.25674071 34.72947677 12.40468444]


In [111]:
predictions_df = pd.DataFrame(predictions)
predictions_df.describe()

,0
count,109.000000
mean,47.289391
std,9.429469
min,17.762046
25%,43.811314
50%,49.810101
75%,53.385499
max,64.107355
